# ML Model Monitoring & Observability

## Why Monitor ML Models?
ML models degrade over time because the real world changes. Unlike traditional software bugs, model failures are silent the model keeps predicting, just poorly.

## Types of Drift

### 1. Data Drift (Covariate Shift)
Input feature distribution changes: $$P_{train}(X) \neq P_{prod}(X)$$

### 2. Concept Drift
The relationship between input and output changes: $$P_{train}(Y|X) \neq P_{prod}(Y|X)$$

### 3. Label Drift
Target distribution changes: $$P_{train}(Y) \neq P_{prod}(Y)$$

### 4. Prediction Drift
Model output distribution changes can detect issues even without labels.

## Statistical Tests for Drift

### Kolmogorov-Smirnov (KS) Test
Measures maximum difference between two CDFs:
$$D = \sup_x |F_1(x) - F_2(x)|$$
Used for: continuous features.

### Population Stability Index (PSI)
$$PSI = \sum_{i=1}^{n} (P_i - Q_i) \ln\frac{P_i}{Q_i}$$
- PSI < 0.1: No significant change
- 0.1 ≤ PSI < 0.2: Moderate change (investigate)
- PSI ≥ 0.2: Significant change (retrain)

### Chi-Square Test
For categorical features: $$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$

### Wasserstein Distance (Earth Mover's Distance)
$$W(P, Q) = \inf_{\gamma \in \Pi(P,Q)} \mathbb{E}_{(x,y)\sim\gamma}[||x-y||]$$

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Simulate training and production data with drift
np.random.seed(42)

# Training distribution
X_train_ref, y_train_ref = make_classification(n_samples=5000, n_features=10, random_state=42)

# Production distribution (drifted shifted means)
X_prod = X_train_ref[:1000].copy()
X_prod[:, 0] += 1.5  # Drift feature 0
X_prod[:, 3] *= 2.0  # Drift feature 3

print('Reference data shape:', X_train_ref.shape)
print('Production data shape:', X_prod.shape)

Reference data shape: (5000, 10)
Production data shape: (1000, 10)


In [2]:
# KS Test for drift detection
def detect_drift_ks(reference, production, feature_names=None, threshold=0.05):
    """Detect drift using Kolmogorov-Smirnov test."""
    n_features = reference.shape[1]
    if feature_names is None:
        feature_names = [f'feature_{i}' for i in range(n_features)]
    
    results = []
    for i, name in enumerate(feature_names):
        ks_stat, p_value = stats.ks_2samp(reference[:, i], production[:, i])
        drifted = p_value < threshold
        results.append({
            'feature': name, 'ks_statistic': ks_stat,
            'p_value': p_value, 'drift_detected': drifted
        })
    return pd.DataFrame(results)

feature_names = [f'feature_{i}' for i in range(10)]
ks_results = detect_drift_ks(X_train_ref, X_prod, feature_names)
print(ks_results.sort_values('ks_statistic', ascending=False).head(5).to_string())

     feature  ks_statistic        p_value  drift_detected
0  feature_0        0.5588  2.137601e-242            True
3  feature_3        0.2156   2.075635e-34            True
6  feature_6        0.0292   4.703314e-01           False
7  feature_7        0.0260   6.199051e-01           False
5  feature_5        0.0226   7.824287e-01           False


In [3]:
# PSI Calculation
def calculate_psi(reference, production, bins=10):
    """Calculate Population Stability Index."""
    # Create bins from reference
    _, bin_edges = np.histogram(reference, bins=bins)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    
    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    prod_counts, _ = np.histogram(production, bins=bin_edges)
    
    # Convert to proportions with smoothing
    eps = 1e-10
    ref_pct = (ref_counts + eps) / (len(reference) + eps * bins)
    prod_pct = (prod_counts + eps) / (len(production) + eps * bins)
    
    psi = np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct))
    return psi

def interpret_psi(psi):
    if psi < 0.1: return 'No change'
    elif psi < 0.2: return 'Moderate change investigate'
    else: return 'SIGNIFICANT CHANGE retrain'

for i in range(10):
    psi = calculate_psi(X_train_ref[:, i], X_prod[:, i])
    print(f'feature_{i}: PSI={psi:.4f} → {interpret_psi(psi)}')

feature_0: PSI=2.5344 → SIGNIFICANT CHANGE retrain
feature_1: PSI=0.0026 → No change
feature_2: PSI=0.0080 → No change
feature_3: PSI=1.1201 → SIGNIFICANT CHANGE retrain
feature_4: PSI=0.0102 → No change
feature_5: PSI=0.0148 → No change
feature_6: PSI=0.0037 → No change
feature_7: PSI=0.0085 → No change
feature_8: PSI=0.0270 → No change
feature_9: PSI=0.0043 → No change


## Evidently AI

Evidently is a Python library for ML monitoring with ready-made reports and tests.

```python
# pip install evidently
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset, RegressionPreset
from evidently.test_suite import TestSuite
from evidently.test_preset import DataStabilityTestPreset, DataDriftTestPreset

# Create DataFrames
ref_df = pd.DataFrame(X_train_ref[:1000], columns=feature_names)
prod_df = pd.DataFrame(X_prod, columns=feature_names)

# Data Drift Report
report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=ref_df, current_data=prod_df)
report.save_html('drift_report.html')

# Test Suite
test_suite = TestSuite(tests=[DataStabilityTestPreset()])
test_suite.run(reference_data=ref_df, current_data=prod_df)
test_suite.save_html('tests.html')

# Get results as dict
results = test_suite.as_dict()
print(f'Tests passed: {results["summary"]["passed_tests"]}')
```

## Prometheus + Grafana for ML Metrics

```python
# pip install prometheus-client fastapi
from prometheus_client import Counter, Histogram, Gauge, start_http_server
import time

# Define metrics
PREDICTION_COUNTER = Counter('ml_predictions_total', 'Total predictions', ['model_version', 'class'])
PREDICTION_LATENCY = Histogram('ml_prediction_latency_seconds', 'Prediction latency', buckets=[.01, .05, .1, .5, 1])
MODEL_ACCURACY = Gauge('ml_model_accuracy', 'Current model accuracy', ['model_version'])
DATA_DRIFT_SCORE = Gauge('ml_data_drift_psi', 'PSI drift score', ['feature'])

# FastAPI integration
from fastapi import FastAPI, Request
app = FastAPI()

@app.post('/predict')
async def predict(request: Request):
    start = time.time()
    data = await request.json()
    
    # prediction = model.predict(data['features'])
    prediction = 1  # placeholder
    
    # Record metrics
    PREDICTION_COUNTER.labels(model_version='v1.2', class=str(prediction)).inc()
    PREDICTION_LATENCY.observe(time.time() - start)
    
    return {'prediction': prediction}

# Expose metrics endpoint
from prometheus_client import make_asgi_app
metrics_app = make_asgi_app()
app.mount('/metrics', metrics_app)
```

## Grafana Dashboard YAML
```yaml
# Alert rule
apiVersion: 1
groups:
  - name: ml_alerts
    rules:
      - alert: ModelAccuracyDrop
        expr: ml_model_accuracy < 0.85
        for: 5m
        labels:
          severity: critical
        annotations:
          summary: 'Model accuracy dropped below 85%'
      
      - alert: HighPredictionLatency
        expr: histogram_quantile(0.99, ml_prediction_latency_seconds) > 0.5
        for: 2m
        annotations:
          summary: 'P99 latency > 500ms'
```

## Additional Learning Resources

### Tools
- [Evidently AI Docs](https://docs.evidentlyai.com/)
- [Prometheus Docs](https://prometheus.io/docs/)
- [Grafana Docs](https://grafana.com/docs/)
- [WhyLogs](https://whylogs.readthedocs.io/) Lightweight logging for distributions

### Books & Courses
- [Designing Machine Learning Systems Chip Huyen](https://www.oreilly.com/library/view/designing-machine-learning/9781098107958/)
- [MLOps Zoomcamp Monitoring module](https://github.com/DataTalksClub/mlops-zoomcamp)
- [Made With ML Monitoring](https://madewithml.com/courses/mlops/monitoring/)

### Papers
- [Failing Loudly: An Empirical Study of Methods for Detecting Dataset Shift](https://arxiv.org/abs/1810.11953)
- [Concept Drift Detection for Streaming Data](https://arxiv.org/abs/1502.07765)